# Turning RAG into Agentic-RAG

In [1]:
from openai import OpenAI

openai_client = OpenAI()

In [2]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

In [3]:
from minsearch import AppendableIndex

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

## Traditional RAG

In [4]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

In [5]:
import json

RAG_INSTRUCTIONS = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

RAG_PROMPT_TEMPLATE = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return RAG_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )


In [ ]:
question = "How do I create a dahsbord in Evidently?"
search_results = search(question)
user_prompt = build_prompt(question, search_results)

In [8]:
messages = [
    {"role": "system", "content": RAG_INSTRUCTIONS},
    {"role": "user", "content": user_prompt}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
)
print(response.output_text)

The provided context does not contain specific instructions on how to create a dashboard in Evidently. Therefore, I am unable to answer your question based on the available information.


## Agentic RAG

In [9]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Use only facts from the knowledge base when answering.
IMPORTANT: f you cannot find the answer, inform the user.
"""

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": [
            "query"
        ]
    }
}


In [ ]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
)

print(response.usage.input_tokens)


110


In [11]:
tool_call = response.output[0]
tool_call

ResponseFunctionToolCall(arguments='{"query":"create dashboard in Evidently"}', call_id='call_f6LErZ6QXidXuEjwI0nowiTx', name='search', type='function_call', id='fc_0d64ce9a8815f4120069f46fa97f208197b0f3930bab0d82e5', namespace=None, status='completed')

In [12]:
messages.append(tool_call)

In [14]:
arguments = json.loads(tool_call.arguments)
arguments

{'query': 'create dashboard in Evidently'}

In [15]:
search_results = search(**arguments)
search_results[:1]

[{'start': 0,
  'content': 'Dashboards let you create Panels to visualize evaluation results over time. Note that to be able to populate the panels, you must first add Reports with evaluation results to the Project.\n\n<Check>\n  No-code Dashboards are available in the Evidently Cloud and Enterprise.\n</Check>\n\n## Adding Tabs\n\nBy default, new Panels appear on a single Dashboard. You can add multiple Tabs to organize them.\n\n**To add a Tab**:\n\n- Enter "Edit" mode on the Dashboard (top right corner).\n- Click the plus sign with "add Tab" on the left.\n- To create a custom Tab, select "empty" and enter a name.\n\nTo simplify setup, you can start with pre-built Tabs. These are dashboard templates with preset Panel combinations:\n\n![Add Dashboard Tab](/images/dashboard/add_dashboard_tab_v2.gif)\n\n**Pre-built Tabs** rely on having related Metrics (or Presets that include the specific Metrics) within the Project. If the necessary data is not available, the Panels will appear empty un

In [16]:
call_output = {
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": json.dumps(search_results),
}

In [17]:
messages.append(call_output)
messages

[{'role': 'system',
  'content': "\nYou're a documentation assistant. \n\nAnswer the user question using the documentation knowledge base\n\nUse only facts from the knowledge base when answering.\nIMPORTANT: f you cannot find the answer, inform the user.\n"},
 {'role': 'user', 'content': 'How do I create a dahsbord in Evidently?'},
 ResponseFunctionToolCall(arguments='{"query":"create dashboard in Evidently"}', call_id='call_f6LErZ6QXidXuEjwI0nowiTx', name='search', type='function_call', id='fc_0d64ce9a8815f4120069f46fa97f208197b0f3930bab0d82e5', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_f6LErZ6QXidXuEjwI0nowiTx',
  'output': '[{"start": 0, "content": "Dashboards let you create Panels to visualize evaluation results over time. Note that to be able to populate the panels, you must first add Reports with evaluation results to the Project.\\n\\n<Check>\\n  No-code Dashboards are available in the Evidently Cloud and Enterprise.\\n</Check>\\n\

In [18]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
)

print(response.output_text)
print(response.usage.input_tokens)

To create a dashboard in Evidently, follow these steps:

### 1. Adding Tabs
- **Enter Edit Mode**: Click the "Edit" button in the top right corner of the Dashboard.
- **Add a Tab**: Click the plus sign to add a new tab. You can either create a custom empty tab or use pre-built templates.

### 2. Adding Panels
You can add various types of panels, including text panels, counters, pie charts, and plots. To add a panel, do the following:
- **Enter Edit Mode**: Again, ensure you are in Edit mode.
- **Click "Add Panel"**: Follow the prompts to configure the panel.
- **Save and Select a Tab**: After setting up the panel, click "Save" and choose the Tab you want to add it to.

#### **Examples of Panel Types**:
- **Text panels**: For titles or important notes.
- **Counters**: To show values like totals or averages.
- **Plots**: To visualize data trends over time.

### 3. Configuration
- **Select Metrics**: Choose the appropriate Metrics that match those in your Reports.
- **Filter By Tags or La